# Risk score

Adds `risk_score` to `regular_season.csv`: for each athlete and season, the share of their races that went wrong.

**risk_score = (DQ + DNF + no-mark + outlier marks) / races started**, counted over the athlete's individual events that season (indoor and outdoor). Every row for that athlete-season gets the same value.

| Counts as a bad race | Codes / rule |
|---|---|
| Disqualified | `DQ`, `FS` (false start) |
| Did not finish | `DNF` |
| No valid field mark | `FOUL`, `NH` (no height), `NM` (no mark): the field-event version of not finishing |
| Outlier mark | A valid mark much worse than the athlete's own season best in that event (see below) |

**Outlier marks.** For every mark, we measure how far off the athlete's season best in that event it was, in %. Normal off-days differ by event: a 100m runner is rarely more than ~3% off, while a javelin thrower is often 10% off. So the cutoff is set per event, from all athletes' marks in that event: **more than 3 interquartile ranges above the 75th percentile** (Tukey's "far out" rule). Examples: about 7% for the 100m, 10% for the 100m hurdles, 32% for the javelin. A mark can only be an outlier if the athlete has at least 2 marks in that event that season.

**Not counted at all:** relays (a DQ or DNF there is a team result), and `DNS`, `NT`, `NP` (the athlete didn't start, or no result was recorded).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path('../data/processed')
REGULAR_SEASON_PATH = DATA_DIR / 'regular_season.csv'

DQ_CODES = {'DQ', 'FS'}
DNF_CODES = {'DNF'}
NO_MARK_CODES = {'FOUL', 'NH', 'NM'}
NOT_STARTED_CODES = {'DNS', 'NT', 'NP'}
RELAY = r'^(?:4x|DMR|SMR)'
OUTLIER_IQRS = 3  # Tukey's "far out" fence

# Read every column as text so the CSV is written back exactly as it was, plus `risk_score`
regular_season = pd.read_csv(REGULAR_SEASON_PATH, dtype=str, keep_default_na=False)
regular_season = regular_season.drop(columns='risk_score', errors='ignore')


def to_number(mark):
    """'10.74' -> 10.74, '3:44.68' -> 224.68, 'DNF'/'FOUL' -> NaN."""
    try:
        return sum(float(part) * 60**i for i, part in enumerate(reversed(str(mark).strip().split(':'))))
    except ValueError:
        return np.nan


races = regular_season[~regular_season['event'].str.match(RELAY)].copy()
races['mark_code'] = races['mark'].str.strip().str.upper()
races = races[~races['mark_code'].isin(NOT_STARTED_CODES)]
races['mark_num'] = races['mark'].map(to_number)
races['lower_is_better'] = races['event_type'].isin(['track', 'unknown'])  # 'unknown' = odd running distances

# % off the athlete's season best in that event (0 = their best, positive = worse)
valid = races['mark_num'].notna()
group = races[valid].groupby(['athlete_id', 'year', 'event'])['mark_num']
season_best = np.where(races.loc[valid, 'lower_is_better'], group.transform('min'), group.transform('max'))
ratio = races.loc[valid, 'mark_num'] / season_best
races.loc[valid, 'pct_off_best'] = np.where(races.loc[valid, 'lower_is_better'], ratio - 1, 1 - ratio) * 100
races.loc[valid, 'n_marks'] = group.transform('size')

# Per-event outlier cutoff, from marks that aren't the athlete's own best
spread = races[(races['n_marks'] >= 2) & (races['pct_off_best'] > 0)].groupby('event')['pct_off_best']
q1, q3 = spread.quantile(0.25), spread.quantile(0.75)
cutoff = (q3 + OUTLIER_IQRS * (q3 - q1)).rename('outlier_cutoff_pct')
races = races.merge(cutoff, left_on='event', right_index=True, how='left')

races['dq'] = races['mark_code'].isin(DQ_CODES)
races['dnf'] = races['mark_code'].isin(DNF_CODES)
races['no_mark'] = races['mark_code'].isin(NO_MARK_CODES)
races['outlier'] = (races['n_marks'] >= 2) & (races['pct_off_best'] > races['outlier_cutoff_pct'])
races['bad_race'] = races[['dq', 'dnf', 'no_mark', 'outlier']].any(axis=1)

risk = (races.groupby(['athlete_id', 'year'])
        .agg(races=('bad_race', 'size'), dq=('dq', 'sum'), dnf=('dnf', 'sum'),
             no_mark=('no_mark', 'sum'), outlier=('outlier', 'sum'), bad_races=('bad_race', 'sum'))
        .reset_index())
risk['risk_score'] = risk['bad_races'] / risk['races']

print(f'{len(races)} individual races, {races["bad_race"].sum()} bad '
      f'(DQ {races["dq"].sum()}, DNF {races["dnf"].sum()}, no mark {races["no_mark"].sum()}, '
      f'outlier {races["outlier"].sum()})')
print(f'{len(risk)} athlete-seasons; risk_score summary:')
print(risk['risk_score'].describe().round(3).to_string())

49832 individual races, 1951 bad (DQ 78, DNF 446, no mark 1227, outlier 200)
5268 athlete-seasons; risk_score summary:
count    5268.000
mean        0.042
std         0.102
min         0.000
25%         0.000
50%         0.000
75%         0.026
max         1.000


In [2]:
out = regular_season.merge(risk[['athlete_id', 'year', 'risk_score']].assign(
                               risk_score=risk['risk_score'].map(lambda r: f'{r:.4f}')),
                           on=['athlete_id', 'year'], how='left', validate='many_to_one')
out['risk_score'] = out['risk_score'].fillna('')  # athlete-seasons with only relays or DNS
assert len(out) == len(regular_season)
out.to_csv(REGULAR_SEASON_PATH, index=False)
print(f'regular_season.csv: risk_score filled for {out["risk_score"].ne("").sum()} of {len(out)} rows')

regular_season.csv: risk_score filled for 57286 of 57300 rows


## Check

In [3]:
print('Outlier cutoffs for the main events (% off season best):')
main = ['100', '200', '400', '800', '1500', '5000', '100H', '110H', '400H', '3000S',
        'HJ', 'PV', 'LJ', 'TJ', 'SP', 'DT', 'HT', 'JT']
display(cutoff.reindex(main).round(1).to_frame().T)

print('Sample of marks flagged as outliers:')
display(races[races['outlier']].sample(12, random_state=0)
        [['athlete_name', 'year', 'event', 'meet_name', 'mark', 'pct_off_best', 'outlier_cutoff_pct']].round(1))

print('Highest risk scores (athlete-seasons with at least 5 races):')
names = races.groupby(['athlete_id', 'year'])['athlete_name'].first()
display(risk[risk['races'] >= 5].join(names, on=['athlete_id', 'year'])
        .sort_values('risk_score', ascending=False).head(10).round(3))

Outlier cutoffs for the main events (% off season best):


event,100,200,400,800,1500,5000,100H,110H,400H,3000S,HJ,PV,LJ,TJ,SP,DT,HT,JT
outlier_cutoff_pct,6.9,7.8,7.8,10.1,9.1,9.2,11.3,9.1,11.3,9.5,13.7,18.4,18.8,14.0,22.8,27.5,23.7,33.2


Sample of marks flagged as outliers:


,athlete_name,year,event,meet_name,mark,pct_off_best,outlier_cutoff_pct
4912,Mariah Stewart,2022,100H,Battle in the OC,20.08,42.4,11.3
51274,Adee Newman,2023,100H,44th Aztec Invitational,18.38,20.6,11.3
37105,Shaelyn Amos,2022,100H,The Vince O'Boyle Track and Field Classic,18.63,14.0,11.3
34523,Jiana Boston,2023,LJ,2023 UW Invite,4.29,21.3,18.8
52472,Phoebe Ladd,2022,LJ,Beach Opener 2022,3.78,28.9,18.8
52700,Quwshayah Peters,2022,400,New Mexico Classic,1:22.76,37.3,7.8
1748,Tammy Vien,2019,100H,UC Irvine Spring Break Collegiate Classic,16.91,15.9,11.3
46389,Alyssa Allen,2019,800,Air Force Invitational,2:35.49,13.6,10.1
3266,Elijah Hodge,2021,200,Cal Poly Blue-Green Rivalry Meet,26.41,19.1,7.8
47406,Allison Foster,2019,LJ,Air Force Invitational,4.49,21.5,18.8


Highest risk scores (athlete-seasons with at least 5 races):


,athlete_id,year,races,dq,dnf,no_mark,outlier,bad_races,risk_score,athlete_name
3463,8360079,2026,6,0,0,4,0,4,0.667,Charlotte O'Connor
3409,8353872,2026,10,0,0,6,0,6,0.600,Mackenzie Okonta
775,6996363,2019,5,0,0,3,0,3,0.600,Nicole Harvard
4419,8789415,2026,5,0,0,3,0,3,0.600,Francesca Villain
563,6666105,2021,5,0,0,3,0,3,0.600,Julia Wang
473,6609597,2021,5,0,0,3,0,3,0.600,Delani Fish
1026,7110102,2019,5,0,0,3,0,3,0.600,Logan Flores
4643,8980607,2025,5,0,0,3,0,3,0.600,Avery Allen
1030,7110104,2019,5,0,0,3,0,3,0.600,Gavin Palmer
2074,7753251,2021,7,0,0,4,0,4,0.571,Samuel Heinrichs
